## Notebook for:
#### - computing the differences between the masks and the predictions
#### - plotting the images with the masks and the predictions, ranked by the noise level

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

root_fp = str(Path('..').resolve())
if not root_fp in sys.path:
    sys.path.insert(0, root_fp)

from src.utils import draw_contours_from_mask

In [ ]:
use_cv = False

all_pred_dirs = []
if use_cv:
    fold_to_test_on = 'test'
    for cv_iter in range(5):
        pred_dir = Path(
            f'../data/external/experiments/refinenet/cv_iter_{cv_iter}/version_0/results/ckpt_first/{fold_to_test_on}')
        assert pred_dir.exists()
        all_pred_dirs.append(pred_dir)
else:
    model_outdir = Path('../data/external/experiments/refinenet/version_1/results/ckpt_best_tta_False_n_1')
    # fold_to_test_on = 'test'
    fold_to_test_on = 'train'
    # fold_to_test_on = 'all'

    if fold_to_test_on == 'all':
        all_pred_dirs = [model_outdir / fold for fold in ['train', 'valid', 'test']]
    else:
        all_pred_dirs = [model_outdir / fold_to_test_on]
for fp in all_pred_dirs:
    assert fp.exists(), f"{fp} does not exist"
print(all_pred_dirs)


In [ ]:
fp_preds = {}
fp_preds_uq = {}

for pred_dir in all_pred_dirs:
    print(f"Reading the results from {pred_dir}")

    # prepare a map from the image id to the prediction path
    fp_preds_crt = sorted(list((pred_dir / 'preds').rglob('*.png')))
    fp_preds.update({fp.name: fp for fp in fp_preds_crt})
    print(f"Found {len(fp_preds_crt)} predictions")

    fp_preds_uq_crt = sorted(list((pred_dir / 'preds_uq').rglob('*.png')))
    fp_preds_uq.update({fp.name: fp for fp in fp_preds_uq_crt})
    print(f"Found {len(fp_preds_uq_crt)} aleatoric uncertainty predictions")
print(f"Found {len(fp_preds)} predictions in total")
print(f"Found {len(fp_preds_uq)} aleatoric uncertainty predictions in total")

# if we use CV, save the combined testing results in a separate directory
if use_cv:
    results_dir = Path(str(all_pred_dirs[0]).replace('cv_iter_0', 'cv_combined'))
    results_dir.mkdir(parents=True, exist_ok=True)
else:
    # results_dir = Path(f'../data/external/experiments/refinenet/version_pretrained/results/test')
    results_dir = model_outdir / fold_to_test_on
print()
print(f"results_dir = {results_dir}")

In [ ]:
ds_dir = Path('../data/external/dataset/training_patches')
fp_list = sorted(list(ds_dir.glob('*.png')))
print(len(fp_list))

print("Keeping only the images for which we have predictions")
fp_list = [fp for fp in fp_list if fp.name in fp_preds]
len(fp_list)

### Compute the differences between the masks and the predictions

In [ ]:
# compute the difference between the mask and the prediction
all_diffs = {k: [] for k in ['fp', 'iou', 'iou_bg', 'iou_balanced', 'acc', 'n_pos_pred', 'n_pos_mask']}

use_morpho = False
for fp_img in tqdm(fp_list):
    img = plt.imread(fp_img) * 255
    img = img.astype(np.uint8)
    mask = (np.array(Image.open(Path(str(fp_img).replace('training_patches', 'training_noisy_labels')))) > 0)

    pred = np.array(Image.open(fp_preds[fp_img.name]))

    if pred.shape[-1] == 3:
        pred = pred[:, :, 0]  # artifact, from the cases where no prediction was made

    if use_morpho:
        kernel_size = 3
        pred = cv2.dilate(pred, np.ones((kernel_size, kernel_size), np.uint8), iterations=1)
        pred = cv2.erode(pred, np.ones((kernel_size, kernel_size), np.uint8), iterations=1)

    pred = (pred > 0)
    den = np.sum(mask | pred)
    iou = 1 - np.sum(mask & pred) / max(den, 1)
    den = np.sum((mask == 0) | (pred == 0))
    iou_bg = 1 - np.sum((mask == 0) & (pred == 0)) / max(den, 1)
    iou_balanced = (iou + iou_bg) / 2
    acc = np.sum(mask == pred) / mask.size
    n_pos_pred = np.sum(pred)
    n_pos_mask = np.sum(mask)
    assert n_pos_mask < (256 ** 2), f"n_pos_mask = {n_pos_mask}"
    assert n_pos_pred < (256 ** 2), f"n_pos_pred = {n_pos_pred}"

    all_diffs['fp'].append(fp_img)
    all_diffs['iou'].append(iou)
    all_diffs['iou_bg'].append(iou_bg)
    all_diffs['iou_balanced'].append(iou_balanced)
    all_diffs['acc'].append(acc)
    all_diffs['n_pos_pred'].append(n_pos_pred)
    all_diffs['n_pos_mask'].append(n_pos_mask)

    # print(f"iou = {iou:.6f}; iou_bg = {iou_bg:.6f}; iou_balanced = {iou_balanced:.6f}; acc = {acc:.6f}; n_pos_pred = {n_pos_pred}; n_pos_mask = {n_pos_mask}")
    # print(f"1 - iou = {1 - iou:.6f}; 1 - iou_bg = {1 - iou_bg:.6f}; 1 - iou_balanced = {1 - iou_balanced:.6f}; 1 - acc = {1 - acc:.6f}")

    if False:
        fig, axes = plt.subplots(1, 3, figsize=(10, 5), dpi=150, facecolor='white')
        img_c = draw_contours_from_mask(img=img / 255, mask=(mask > 0), color=(1, 0, 0), thickness=1, alpha=0.9)
        axes[0].imshow(img_c)
        axes[0].set_title('Ground truth')
        axes[1].imshow(pred, cmap='gray')
        axes[1].set_title('Prediction')
        diff = (mask != pred)
        axes[2].imshow(diff * 255, cmap='gray')
        axes[2].set_title(f'Difference = {diff.sum()}')
        for ax in axes:
            ax.axis('off')
        plt.show()
        break


In [ ]:
df_out = pd.DataFrame(all_diffs)
df_out['f_pos_pred'] = df_out['n_pos_pred'] / (256 ** 2)
df_out['f_pos_mask'] = df_out['n_pos_mask'] / (256 ** 2)
df_out['image_id'] = df_out['fp'].apply(lambda x: x.name)
df_out = df_out.sort_values('image_id')

# save all the stats
results_dir.mkdir(parents=True, exist_ok=True)
label = f"results_morpho_{use_morpho}"
fp_out = results_dir / f'{label}_extra.csv'
df_out.to_csv(fp_out, index=False)
print(f"Saved the extra results to {fp_out}")

# save the results in Kaggle format
noise_metric = 'iou'
# noise_metric = 'iou_balanced'
# noise_metric = 'acc'
fp_out = results_dir / f'{label}_metric_{noise_metric}.csv'
df_out['noise_level'] = df_out[noise_metric]
df_out = df_out.sort_values(['noise_level', 'image_id'])  # use the image_id to break the ties
df_out['id'] = np.arange(len(df_out))
df_out[['id', 'image_id']].to_csv(fp_out, index=False)
print(f"Saved the results in Kaggle format to {fp_out}")

df_out

### Plot the images with the masks and the predictions, ranked by the noise level in batches of 120

In [ ]:
noise_metric = 'iou'
# noise_metric = 'iou_balanced'
use_morpho = False
fp_out_results = results_dir / f"results_morpho_{use_morpho}_metric_{noise_metric}.csv"
print(f"Reading the results from {fp_out_results}")
df_out = pd.read_csv(fp_out_results)
df_out_extra = pd.read_csv(results_dir / f'results_morpho_{use_morpho}_extra.csv')
df_out_extra

In [ ]:
print(f"results_dir = {results_dir}")
i_batch_list = list(range(0, int(np.ceil(len(df_out) / 120))))
# i_batch_list = [0]
# i_batch_list = [-1]
# i_batch_list = [0, 1]
print(i_batch_list)

fp_list_d = {fp.name: fp for fp in fp_list}
# plot also the aleatoric uncertainty if available
# for sort_by in ['image_id', 'noise_level']:
for sort_by in ['image_id']:
    crt_df_out = df_out if sort_by == 'noise_level' else df_out.sort_values('image_id')

    # add the noise level
    crt_df_out = crt_df_out.merge(df_out_extra)
    crt_df_out['noise_level'] = crt_df_out[noise_metric]
    for plot_type in ['pred', 'pred_uq']:
        if plot_type == 'pred_uq' and len(fp_preds_uq) == 0:
            print("No aleatoric uncertainty predictions available. Skipping...")
            continue
        crt_fp_preds = fp_preds if plot_type == 'pred' else fp_preds_uq
        for i_batch in tqdm(i_batch_list):
            i_start = i_batch * 120
            i_end = None if i_batch == -1 else max((i_batch + 1) * 120, len(crt_df_out))
            sdf = crt_df_out.iloc[i_start:i_end]
            fig, axs = plt.subplots(8, 15, figsize=(40, 20))
            for i, ax in enumerate(axs.flat):
                # stop if we reached the end
                if i >= len(sdf):
                    break

                # prepare the paths
                fp_img = Path(sdf.iloc[i].fp)
                assert fp_img.exists()
                pred_fp = crt_fp_preds[fp_img.name]
                assert pred_fp.exists()
                mask_fp = Path(str(fp_img).replace('training_patches', 'training_noisy_labels'))
                assert mask_fp.exists()

                # read the mask
                mask = np.array(Image.open(mask_fp))

                if plot_type == 'pred':  # read the original image and draw the predictions on top of it
                    img = np.array(Image.open(fp_img)) / 255.0

                    # draw the predictions on top of the image
                    pred = np.array(Image.open(pred_fp))
                    img_c = draw_contours_from_mask(img=img, mask=(pred > 0), color=(1, 0, 0), thickness=1, alpha=1.0)
                else:
                    # plot the aleatoric uncertainty only
                    img_c = np.array(Image.open(pred_fp).convert('RGB')) / 255.0

                # extract the contours of the mask and draw them on top of the image in blue
                img_c = draw_contours_from_mask(img=img_c, mask=(mask > 0), color=(0, 0, 1), thickness=1, alpha=1.0)

                ax.imshow(img_c)
                ax.set_title(f"i = {fp_list.index(fp_img)}; noise = {sdf.noise_level.iloc[i] * 100:.2f} %", fontsize=8)
                ax.axis('off')
            plt.tight_layout()

            if len(i_batch_list) == 1:
                plt.show()
                break

            fp_out = results_dir / 'plots' / f"sorted_by_{sort_by}_{fp_out_results.stem}" / plot_type / f'{results_dir.name}_batch_{i_batch:02d}.png'
            fp_out.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(fp_out)
            plt.close('all')